# 00 — Verify Connection

Confirms three things before the pipeline can run:

1. **Spark / Databricks Connect** session is live (or you're inside a workspace where `spark` already exists)
2. **Unity Catalog target schemas** exist — creates `{UC_SCHEMA}_bronze`, `_silver`, `_gold` if needed
3. **rostr.cc API** credentials work — performs a single login probe and one artist lookup

Run this first. Every other notebook in the demo expects the schemas, the Volume, and a working rostr session.

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'rostr_music_industry')
BRONZE_SCHEMA = f'{UC_SCHEMA}_bronze'
SILVER_SCHEMA = f'{UC_SCHEMA}_silver'
GOLD_SCHEMA   = f'{UC_SCHEMA}_gold'
VOLUME_NAME   = 'raw_data'
VOLUME_PATH   = f'/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}'

print(f'Catalog: {UC_CATALOG}')
print(f'Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}')
print(f'Silver:  {UC_CATALOG}.{SILVER_SCHEMA}')
print(f'Gold:    {UC_CATALOG}.{GOLD_SCHEMA}')
print(f'Volume:  {VOLUME_PATH}')

In [ ]:
# Spark + workspace identity
spark.sql('SELECT current_user(), current_timestamp(), current_catalog()').show(truncate=False)
print('Spark version:', spark.version)

In [ ]:
# Create the three medallion schemas + the raw_data Volume on bronze
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}')
    print(f'  schema ready: {UC_CATALOG}.{schema}')

spark.sql(f'CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}')
print(f'  volume ready: {VOLUME_PATH}')

## rostr.cc API probe

POSTs `email + password` to `/v1/auth/rostr` (no headless browser needed —
the SPA's Castle/Turnstile checks aren't enforced on the API), caches the
session cookie, then exercises the artist + team endpoints.

In [ ]:
def get_secret(env_var: str, scope: str = 'rostr', key: str | None = None) -> str:
    """Read a secret from .env first, then fall back to a Databricks secret scope.

    Lets the same notebook run from a laptop (.env) or inside a Databricks
    Git Folder (workspace secret scope `rostr`).
    """
    val = os.getenv(env_var)
    if val:
        return val
    try:
        from pyspark.dbutils import DBUtils  # type: ignore
        dbutils = DBUtils(spark)
        return dbutils.secrets.get(scope=scope, key=key or env_var.lower())
    except Exception as e:
        raise RuntimeError(
            f"Could not resolve {env_var}: not in environment and Databricks "
            f"secret scope '{scope}' lookup failed ({e})."
        )

In [ ]:
from rostr_client import RostrClient

rostr = RostrClient(
    username=get_secret('ROSTR_USERNAME'),
    password=get_secret('ROSTR_PASSWORD'),
    cookie_path=os.getenv('ROSTR_COOKIE_PATH') or None,
)
rostr.authenticate()
print('Login OK')

team = rostr.lookup_artist_team('Olivia Rodrigo')
print(f'  rostrId      : {team.rostr_id}')
print(f'  agency rows  : {len(team.by_role("AGENCY"))}')
print(f'  mgmt rows    : {len(team.by_role("MANAGEMENT"))}')
for c in team.contacts[:5]:
    print(f'    {c.role:<10s} {c.company:<25s} {c.person_name or ""}')

## Optional reset

Uncomment to wipe the demo schemas (preserves the Volume so you don't have to re-pull data).

In [ ]:
# for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
#     spark.sql(f'DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE')
#     spark.sql(f'CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}')
# print('Schemas reset.')